# Autoencoder PyTorch — vitais (Limen Épico 9 / E9.3)

> O Limen é um protótipo acadêmico e **não** é um dispositivo médico.

Evidência de treino com **epochs**, **loss** e **early stopping**.
O runtime (`VitalsAnomalyEngine`) **não** carrega este AE — ver
[ADR 0029](../docs/adr/0029-vitais-ml-hibrido.md).

**CI não executa este notebook** e a imagem da API **não** instala PyTorch.

## Ambiente ML (fora do backend)

```bash
python -m venv .venv-ml && source .venv-ml/bin/activate
pip install -r notebooks/requirements-ml.txt
jupyter notebook notebooks/train_vitals_autoencoder.ipynb
```

Dados: `data/processed/vitals/train_features.csv` (ETL) ou fallback das
fixtures sintéticas (`data/fixtures/vitals/`).

In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("..").resolve()
PROCESSED = ROOT / "data" / "processed" / "vitals" / "train_features.csv"
FIXTURES = ROOT / "data" / "fixtures" / "vitals"
EXPORT_PATH = ROOT / "models" / "vitals" / "ae_export.pt"

FEATURE_COLUMNS = (
    "heart_rate",
    "spo2",
    "systolic_bp",
    "diastolic_bp",
    "respiratory_rate",
)
SEED = 20260721
MAX_EPOCHS = 40
PATIENCE = 5
BATCH_SIZE = 32
LR = 1e-3
EXPORT_WEIGHTS = False  # True para gravar ae_export.pt (gitignored)

torch.manual_seed(SEED)
np.random.seed(SEED)
print("ROOT:", ROOT)

In [ ]:
def load_feature_rows() -> list[dict[str, str]]:
    if PROCESSED.is_file():
        with PROCESSED.open(newline="", encoding="utf-8") as fh:
            rows = list(csv.DictReader(fh))
        print(f"Loaded {len(rows)} rows from {PROCESSED.relative_to(ROOT)}")
        return rows
    rows: list[dict[str, str]] = []
    for path in sorted(FIXTURES.glob("vitals_*.csv")):
        with path.open(newline="", encoding="utf-8") as fh:
            rows.extend(csv.DictReader(fh))
    print(f"Fallback fixtures: {len(rows)} rows")
    return rows


rows = load_feature_rows()
assert rows, "Sem dados — rode scripts/etl_vitals_datasets.py --from-fixtures"

# Treina preferencialmente em pontos "normal" (reconstrução); holdout misto.
normals = [r for r in rows if r.get("label", "normal") == "normal"]
train_src = normals if len(normals) >= 20 else rows
X = np.asarray(
    [[float(r[c]) for c in FEATURE_COLUMNS] for r in train_src],
    dtype=np.float32,
)
mu, sigma = X.mean(axis=0), X.std(axis=0)
sigma = np.where(sigma < 1e-6, 1.0, sigma)
X_norm = (X - mu) / sigma

n = len(X_norm)
idx = np.random.permutation(n)
split = max(1, int(0.8 * n))
train_x = torch.from_numpy(X_norm[idx[:split]])
val_x = torch.from_numpy(X_norm[idx[split:]])
train_loader = DataLoader(TensorDataset(train_x), batch_size=BATCH_SIZE, shuffle=True)
print(f"train={len(train_x)} val={len(val_x)} dim={X.shape[1]}")

In [ ]:
class VitalsAutoencoder(nn.Module):
    """AE compacto — só evidência de notebook (fora do worker)."""

    def __init__(self, dim: int = 5, hidden: int = 8, latent: int = 3) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, latent),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden),
            nn.ReLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


model = VitalsAutoencoder(dim=X.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

history_train: list[float] = []
history_val: list[float] = []
best_val = float("inf")
best_state = None
stale = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total = 0.0
    n_batches = 0
    for (batch,) in train_loader:
        opt.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred, batch)
        loss.backward()
        opt.step()
        total += float(loss.item())
        n_batches += 1
    train_loss = total / max(n_batches, 1)

    model.eval()
    with torch.no_grad():
        val_loss = float(loss_fn(model(val_x), val_x).item()) if len(val_x) else train_loss

    history_train.append(train_loss)
    history_val.append(val_loss)
    print(f"epoch={epoch:02d} train_loss={train_loss:.5f} val_loss={val_loss:.5f}")

    if val_loss + 1e-6 < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f"Early stopping at epoch={epoch} (patience={PATIENCE})")
            break

assert len(history_train) >= 1, "Esperado ≥1 epoch registrada"
if best_state is not None:
    model.load_state_dict(best_state)
print(f"Epochs run={len(history_train)} best_val={best_val:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, len(history_train) + 1), history_train, label="train loss")
ax.plot(range(1, len(history_val) + 1), history_val, label="val loss")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Vitals autoencoder — learning curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Export opcional de pesos (gitignored). Não entra no worker/API.
if EXPORT_WEIGHTS:
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "feature_columns": list(FEATURE_COLUMNS),
            "mu": mu.tolist(),
            "sigma": sigma.tolist(),
            "seed": SEED,
            "epochs_run": len(history_train),
            "best_val_loss": best_val,
        },
        EXPORT_PATH,
    )
    print(f"Exported ae_export.pt → {EXPORT_PATH}")
else:
    print("Export desligado (EXPORT_WEIGHTS=False). Ligue para gravar models/vitals/ae_export.pt.")

## Notas

- Comparação limiares vs Isolation Forest vs AE fecha no Épico 9.4.
- `LIMEN_VITALS_BACKEND` **não** aceita `autoencoder`.
- Regenerar features: `python scripts/etl_vitals_datasets.py --from-fixtures`.